# Fine tuning modele Qwen2-VL-2B-Instruct-bnb-4bit

## Installation of libraries

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    # Install unsloth and its core dependencies without accelerate
    !pip install --no-deps unsloth_zoo bitsandbytes {xformers} peft trl triton unsloth
    # Explicitly install a recent version of accelerate with its dependencies
    !pip install accelerate>=0.30.0
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2
# mlflow and dagshub
!pip install mlflow dagshub
# jiwer and rapidfuz
!pip install jiwer rapidfuzz

## Load model unsloth

In [4]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-09 19:54:50.821576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778356491.014232     102 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778356491.073676     102 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778356491.558059     102 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778356491.558114     102 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778356491.558117     102 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

We now add LoRA adapters for parameter efficient finetuning - this allows

---

us to only efficiently train 1% of all parameters.


In [ ]:
peft_model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 32,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

In [ ]:
peft_model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290


## *Data* preparation

To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "user",
  "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
},
{ "role": "assistant",
  "content": [{"type": "text",  "text": A} ]
},
]
```

In [5]:
# Configuration
image_folder = "/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/"
json_folder = "/kaggle/input/datasets/ahmadoubg/dataset-receipt/dataJSON/"
output_path = "dataset.jsonl"

In [6]:
import os
import json
from pathlib import Path

def build_dataset(image_folder, json_folder, output_path):

    dataset = []

    # Correctly iterate over files in the image_folder
    for img_filename in os.listdir(image_folder):
        # Ensure it's a file and not a directory
        if not os.path.isfile(os.path.join(image_folder, img_filename)):
            continue

        base_name = Path(img_filename).stem
        json_path = os.path.join(json_folder, base_name + '.json')

        if not os.path.exists(json_path):
            continue

        with open(json_path, 'r') as f:
            json_data = json.load(f)

        # Optional validation
        if not all(k in json_data for k in ["company", "date", "address", "total"]):
            continue

        conversation = {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": os.path.join(image_folder, img_filename)},
                        {
                            "type": "text",
                            "text": (
                                "Extract company, date, address and total from this receipt.\n"
                                "If missing return null.\n"
                                "Do NOT guess.\n"
                                "Return ONLY valid JSON."
                            )
                        }
                    ]
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": json.dumps(json_data, ensure_ascii=False)}
                    ]
                }
            ]
        }

        dataset.append(conversation)

    # Save JSONL
    with open("dataset_format.jsonl", "w") as f:
        for item in dataset:
            f.write(json.dumps(item) + "\n")

    print(f"✅ Saved {len(dataset)} samples")

    return dataset

In [7]:
dataset = build_dataset(image_folder, json_folder, output_path)

✅ Saved 949 samples


In [31]:
display(dataset[0]['messages'][1]['content'][0]['text'])

'{"company": "B & BEST RESTAURANT", "date": "22/04/2017", "address": "NO.12,JALAN SS4C/5,PETALING JAYA SELANGOR DARUL EHSAN", "total": "22.25"}'

In [9]:
dataset[0]['messages'][0]['content'][0]['image']


'/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/623.jpg'

## Evaluation

### evaluation using metrics:

**Accuracy, Recall, Precision, F1 score and CER**

In [10]:
import re
import numpy as np
from jiwer import cer
from rapidfuzz import fuzz

# ---------------------------
# Config
# ---------------------------
FIELDS = ["company", "date", "address", "total"]

WEIGHTS = {
    "company": 1.0,
    "date": 2.0,
    "address": 1.0,
    "total": 3.0
}

FUZZY_FIELDS = ["company", "address"]
EXACT_FIELDS = ["date", "total"]

### Fonctions helpers

In [11]:
def safe_json_extract(text):
    """Extract JSON safely from model output"""
    try:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    return None


def normalize_text(x):
    if x is None:
        return ""
    return str(x).strip().lower()


def is_valid_total(x):
    return bool(re.match(r'^\d+(\.\d{1,2})?$', x))


def is_valid_date(x):
    return bool(re.match(r'^\d{2}[/-]\d{2}[/-]\d{2,4}$', x))


def fuzzy_match(a, b, threshold=85):
    return fuzz.ratio(a, b) >= threshold

### Fonction eval

In [12]:
def evaluate_sample(gt_str, pred_str):
    # CER global (optional)
    global_cer = cer(gt_str.lower(), pred_str.lower())

    gt_dict = safe_json_extract(gt_str)
    pred_dict = safe_json_extract(pred_str)

    if gt_dict is None or pred_dict is None:
        return {
            "valid_json": 0,
            "accuracy": 0,
            "precision": 0,
            "recall": 0,
            "f1": 0,
            "cer": global_cer,
            "field_cer": 0 # Added to prevent KeyError
        }

    tp, fp, fn = 0, 0, 0
    weighted_correct = 0
    total_weight = sum(WEIGHTS.values())

    field_cers = []

    for field in FIELDS:
        gt_v = normalize_text(gt_dict.get(field, ""))
        pred_v = normalize_text(pred_dict.get(field, ""))

        # --- CER per field ---
        if gt_v and pred_v:
            field_cers.append(cer(gt_v, pred_v))

        match = False

        # --- Matching logic ---
        if field in FUZZY_FIELDS:
            match = fuzzy_match(gt_v, pred_v)

        elif field in EXACT_FIELDS:
            if field == "total":
                if is_valid_total(pred_v):
                    match = (gt_v == pred_v)

            elif field == "date":
                if is_valid_date(pred_v):
                    match = (gt_v == pred_v)

        # --- Metrics update ---
        if gt_v:
            if match:
                tp += 1
                weighted_correct += WEIGHTS[field]
            elif not pred_v:
                fn += 1
            else:
                fp += 1
        else:
            if pred_v:
                fp += 1
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    weighted_accuracy = weighted_correct / total_weight
    avg_field_cer = np.mean(field_cers) if field_cers else 0

    return {
        "valid_json": 1,
        "accuracy": weighted_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "cer": global_cer,
        "field_cer": avg_field_cer
     }

### Initialistion of dagshub and mflow for save experiments

In [16]:
import dagshub
import mlflow

# Remplace par tes vrais identifiants DagsHub
dagshub.init(repo_owner="AhmadouBG", repo_name="qwen2-VL-2B-receipts", mlflow=True)
mlflow.set_experiment("Qwen2-VL-2B-Instruct-bnb-4bit_Evaluation")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a579647a-fd43-412f-a6af-b2441da20e60&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e4138803ab2d1ce3e07242be69aa5f1db19dde3197da039234e69bfc2570bb78




Accessing as AhmadouBG

Initialized MLflow to track repo "AhmadouBG/qwen2-VL-2B-receipts"

Repository AhmadouBG/qwen2-VL-2B-receipts initialized!

2026/05/09 20:02:05 INFO mlflow.tracking.fluent: Experiment with name 'Qwen2-VL-2B-Instruct-bnb-4bit_Evaluation' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/f2f89f705d01455ca792bd2cbd5e9fae', creation_time=1778356925121, experiment_id='0', last_update_time=1778356925121, lifecycle_stage='active', name='Qwen2-VL-2B-Instruct-bnb-4bit_Evaluation', tags={}, trace_location=None, workspace='default'>

Before fine tuning, let's check the first example for the model output using 20 sample and 10 sample after fine tuned with unseen data.

In [41]:
import io
from PIL import Image
import base64

def run_and_log_evaluation(model, dataset_subset, n_sample, run_name="Evaluation"):
    model.eval()
    all_metrics = []
    json_invalid_count = 0
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "model_type": "Qwen2-VL-2B-Instruct-bnb-4bit",
            "temperature": 0.0001,
            "status": "post-finetuning" if hasattr(model, "peft_config") else "base"
        })

        for i in range(min(len(dataset_subset), n_sample)): # Augmenté à 20 pour plus de fiabilité
            example = dataset_subset[i]
            print(example)
            image_pil = Image.open(example['messages'][0]['content'][0]['image']).convert("RGB")

            messages_for_template = [example['messages'][0]]

            # Template propre
            input_text = tokenizer.apply_chat_template(messages_for_template, add_generation_prompt=True)

            inputs = tokenizer(image_pil,
                               input_text,
                               add_special_tokens = False,
                                return_tensors="pt",
                               ).to("cuda")

            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.0001, do_sample=False)

            # Nettoyage strict
            generated_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            # On ne garde que ce qui vient APRÈS le prompt
            generated_response = generated_response.split("Assistant:")[-1].strip()
            print("generated_response", generated_response)

            # Extraction de la réponse JSON
            # Extraction GT
            gt_json_str = example['messages'][1]['content'][0]['text']

            # Évaluation avec sécurité
            try:
                res = evaluate_sample(gt_json_str, generated_response)
                if not res.get('valid_json', False): json_invalid_count += 1
                all_metrics.append(res)
            except Exception as e:
                print(f"Erreur évaluation index {i}: {e}")

            image_pil.close() # Libérer la RAM

        # Métriques agrégées
        metrics_to_log = {
            "avg_precision": np.mean([m['precision'] for m in all_metrics]),
            "avg_recall": np.mean([m['recall'] for m in all_metrics]),
            "avg_f1": np.mean([m['f1'] for m in all_metrics]),
            "avg_cer": np.mean([m['cer'] for m in all_metrics]),
            "json_failure_rate": json_invalid_count / len(all_metrics) if all_metrics else 1
        }

        mlflow.log_metrics(metrics_to_log)
        print(f"✅ Terminé. F1: {metrics_to_log['avg_f1']:.2f} | JSON Fail: {metrics_to_log['json_failure_rate']:.2%} | CER: {metrics_to_log['avg_cer']:.2f} | Precision: {metrics_to_log['avg_precision']:.2f} | Recall: {metrics_to_log['avg_recall']:.2f}")


In [33]:
run_and_log_evaluation(model, dataset, n_sample=20, run_name="Base_Model_Test")

{'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/623.jpg'}, {'type': 'text', 'text': 'Extract company, date, address and total from this receipt.\nIf missing return null.\nDo NOT guess.\nReturn ONLY valid JSON.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '{"company": "B & BEST RESTAURANT", "date": "22/04/2017", "address": "NO.12,JALAN SS4C/5,PETALING JAYA SELANGOR DARUL EHSAN", "total": "22.25"}'}]}]}
generated_response system
You are a helpful assistant.
user
Extract company, date, address and total from this receipt.
If missing return null.
Do NOT guess.
Return ONLY valid JSON.
assistant
{
  "company": "B & Best Restaurant",
  "date": "22/04/2017",
  "address": "NO.12, JALAN SS4C/5, PETALING JAYA SELANGOR DARUL EHSAN",
  "total": 22.25
}
{'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/764.jpg'}, {'type': 'text

Configuration of image in the processor

In [ ]:
tokenizer.image_processor.do_resize = True
tokenizer.image_processor.size = {"shortest_edge": 768}
tokenizer.image_processor.do_center_crop = False

Log of hugging face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('checkpoint')
login(token=token)


Split data

In [ ]:
from datasets import Dataset

# Convert the list to a Dataset object
hf_dataset = Dataset.from_list(dataset)

split_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

### Supervised Fine-tuning (SFT)

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    eval_dataset=test_dataset,  # 🔥 ADD THIS
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        num_train_epochs=5,  # 🔥 better than max_steps

        learning_rate=5e-5,
        warmup_ratio=0.03,

        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,

        optim="adamw_torch",
        weight_decay=0.01,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        seed=3407,
        output_dir="outputs",
        report_to="none",

        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,

        push_to_hub=True,               # Automatic send
        hub_model_id="gueye07/Qwen-Receipt-FineTuned", # name
        hub_strategy="checkpoint",
    ),
)

Unsloth: Model does not have a default image size - using 512


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
5.699 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 854 | Num Epochs = 5 | Total steps = 535
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,358,144 of 2,213,343,744 (0.20% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.140300,0.128053
200,0.042500,0.037976
300,0.028000,0.034229
400,0.031000,0.031564
500,0.035200,0.030522


Unsloth: Not an error, but Qwen2VLForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## Load fine tuned model

In [36]:
from unsloth import FastVisionModel
import torch

# 1. Charger votre adaptateur depuis Hugging Face
# Unsloth va automatiquement télécharger le modèle de base Qwen2-VL-2B
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "gueye07/Qwen-Receipt-FineTuned",
    load_in_4bit = False # On charge en 4bit pour économiser la RAM sur Kaggle
)

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

### Test with unseen data

In [37]:
image_folder = "/kaggle/input/datasets/ahmadoubg/dataset-unseen/image_unseen/"
json_folder = "/kaggle/input/datasets/ahmadoubg/dataset-unseen/dto/"
output_path = "dataset.jsonl"

In [40]:
dataset_test = build_dataset(image_folder, json_folder, output_path)


✅ Saved 17 samples


In [42]:
run_and_log_evaluation(model, dataset_test, run_name="Base_Model_Fine_Tuned")

{'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/kaggle/input/datasets/ahmadoubg/dataset-unseen/image_unseen/950.jpg'}, {'type': 'text', 'text': 'Extract company, date, address and total from this receipt.\nIf missing return null.\nDo NOT guess.\nReturn ONLY valid JSON.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '{"company": "KEDAI PAPAN YEW CHUAN", "date": "11/05/2018", "address": "LOT 276 JALAN BANTING 43800 DENGKIL, SELANGOR", "total": "68.90"}'}]}]}
generated_response system
You are a helpful assistant.
user
Extract company, date, address and total from this receipt.
If missing return null.
Do NOT guess.
Return ONLY valid JSON.
assistant
{"company": "KEDAI PAPAN YEW CHUAN", "date": "11/05/2018", "address": "LOT 276 JALAN BANTING 43800 DENGKIL, SELANGOR.", "total": "68.90"}
{'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/kaggle/input/datasets/ahmadoubg/dataset-unseen/image_unseen/951.jpg'}, {'type': 'text', 'text': 

## Saving to float16 for VLLM
We also support saving to `float16` directly. Select `merged_16bit` for float16.

### Check disk usage and clear space

If your `/kaggle/working` directory is full, the following commands can help you identify and optionally remove large files. **Be careful when deleting files, ensure you're removing files you no longer need.**

### Save the model in hugging face space

In [ ]:
if True: model.push_to_hub_merged("gueye07/qwen2_unsloth_finetune", tokenizer, token = "hf_REDACTED_TOKEN_1234567890abcdefghijklm")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/4.42G [00:00<?, ?B/s]

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:10<00:00, 70.67s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/gueye07/qwen2_unsloth_finetune`


In [ ]:
# Check disk space
!df -h /kaggle/working

# List contents of /kaggle/working to see large files (sort by size)
!ls -lh /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   18G  2.3G  89% /kaggle/working
total 8.0G
-rw-r--r--  1 root root 707M Apr 29 13:29 dataset_format.json
-rw-r--r--  1 root root 503K Apr 29 13:55 dataset_format.jsonl
-rw-r--r--  1 root root 537K Apr 29 13:05 dataset.jsonl
drwxr-xr-x 27 root root 4.0K Apr 29 17:01 llama.cpp
drwxr-xr-x  8 root root 4.0K Apr 29 15:38 outputs
drwxr-xr-x  3 root root 4.0K Apr 29 16:17 qwen2_full_16bit
drwxr-xr-x  3 root root 4.0K Apr 29 16:53 qwen2_gguf_receipt
drwxr-xr-x  2 root root 4.0K Apr 29 15:47 qwen2_lora_receipt
-rw-r--r--  1 root root 2.9G Apr 29 16:57 qwen_model_F16.gguf
-rw-r--r--  1 root root 2.9G Apr 29 16:59 qwen_model_receipt_F16.gguf
-rw-r--r--  1 root root 1.6G Apr 29 17:02 qwen_model_receipt_q8_0.gguf
drwxr-xr-x  4 root root 4.0K Apr 29 13:27 unsloth_compiled_cache


In [ ]:
!rm -rf llama.cpp/

If you need to delete files, you can use `!rm -rf filename` (replace `filename` with the actual file or directory you want to delete). For example, to remove the `qwen2_gguf_receipt` folder if it contains old large files:

```bash
!rm -rf qwen2_gguf_receipt
```

Once you have cleared sufficient space, you can proceed with saving your GGUF models. The existing cell `Q9goXdnACg2Y` converts to `q8_0`. Here's how to save to `f16` (float16) GGUF:

## Install llama.cpp for convertion in gguf model

In [34]:
!apt-get update
!apt-get install pciutils build-essential cmake curl libcurl4-openssl-dev -y
!git clone https://github.com/ggml-org/llama.cpp

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [91.2 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,615 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-driver

In [35]:
%cd llama.cpp
!cmake -B build -DBUILD_SHARED_LIBS=OFF -DGGML_CUDA=OFF # Changed DGGML_CUDA=ON to OFF to avoid CUDA linking error
!cmake --build build --config Release -j 12 --clean-first --target llama-quantize llama-cli llama-mtmd-cli llama-server llama-gguf-split
!cp build/bin/llama-* .
!ls -l .

/kaggle/working/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU b

### Load lora adapter

In [ ]:
from unsloth import FastVisionModel
import torch

# 1. Charger votre adaptateur depuis Hugging Face
# Unsloth va automatiquement télécharger le modèle de base Qwen2-VL-2B
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "gueye07/Qwen-Receipt-FineTuned",
    load_in_4bit = False # On charge en 4bit pour économiser la RAM sur Kaggle
)



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-03 23:12:40.685821: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777849960.916566     105 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777849960.985601     105 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777849961.505793     105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777849961.505822     105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777849961.505825     105 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.4.8: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

In [ ]:
# 2. Sauvegarder en mode FUSIONNÉ (Merged) 16-bit
# C'est cette étape qui crée un modèle complet "prêt à l'emploi"
model.save_pretrained_merged("qwen2_receipt_full", tokenizer, save_method = "merged_16bit")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `qwen2_receipt_full`: 100%|██████████| 1/1 [00:02<00:00,  2.70s/it]


Successfully copied all 1 files from cache to `qwen2_receipt_full`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:21<00:00, 21.64s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/llama.cpp/qwen2_receipt_full`


In [ ]:
# Use the path where you saved your merged model
model, tokenizer =  FastVisionModel.from_pretrained(
    model_name = "/kaggle/working/llama.cpp/qwen2_receipt_full",
    max_seq_length = 2048, # Use the same length used during training
    dtype = None,          # Auto-detect
    load_in_4bit = True,   # Use 4bit for loading to save memory
)

==((====))==  Unsloth 2026.4.8: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### Convert in gguf in q8_0

It seems the previous cells for saving the model in Hugging Face format and converting it to GGUF were not executed. These steps are crucial for creating the necessary GGUF files that `llama-cli` needs. I will generate these steps again now, along with a check for the `mmproj` file.

In [ ]:
%cd /kaggle/working/

# On s'assure que le répertoire de destination existe
!python llama.cpp/convert_hf_to_gguf.py llama.cpp/qwen2_receipt_full \
    --outfile qwen2_model_receipt_q8_0.gguf --outtype q8_0



/kaggle/working
INFO:hf-to-gguf:Loading model: qwen2_receipt_full
INFO:hf-to-gguf:Model architecture: Qwen2VLForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> Q8_0, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> Q8_0, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> Q8_0, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> Q8_0, shape = {1536, 256}
INFO:hf-t

In [ ]:
# Now, modify the llama-cli command to use full paths
# Assuming llama-cli is at /kaggle/working/llama.cpp/llama-cli
# Assuming main GGUF model is at /kaggle/working/qwen2_model_receipt_q8_0.gguf
# Assuming mmproj model is at /kaggle/working/llama.cpp/qwen2_receipt_full/mmproj_qwen2_model_receipt_f16.gguf

# First, change to the llama.cpp directory to ensure llama-cli can be executed directly


!./llama-cli \
    -m /kaggle/working/qwen2_model_receipt_q8_0.gguf \
    --mmproj /kaggle/working/mmproj_qwen2_model_receipt_f16.gguf \
    --image /kaggle/input/datasets/ahmadoubg/dataset-receipt/images/001.jpg \
    -p "Extract company, date, address and total from this receipt.\nIf missing return null.\nDo NOT guess.\nReturn ONLY valid JSON."


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/- 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b9013-e48034dfc
model      : qwen2_model_receipt_q8_0.gguf
modalities : text, vision

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern
  /image <file>       add an image file

Loaded media from '/kaggle/input/datasets/ahmadoubg/dataset-receipt/images/001.jpg'

> Extract company, date, address and total from this receipt.
If missing return null.
Do NOT guess.
Return ONLY valid JSON.

|-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|

### Load the merged model

In [ ]:
# Installer Git LFS
!git lfs install

# Cloner votre dépôt (cela va prendre quelques minutes, ~4-5 Go)
!git clone https://huggingface.co/gueye07/qwen2_unsloth_finetune


Git LFS initialized.
Cloning into 'qwen2_unsloth_finetune'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 22 (delta 0), reused 0 (delta 0), pack-reused 19 (from 1)
Receiving objects: 100% (22/22), 1.72 MiB | 24.46 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Encountered 1 file(s) that may not have been copied correctly on Windows:
	model.safetensors

See: `git lfs help smudge` for more details.


In [ ]:
%cd /kaggle/working/
!sed -i 's/raise NotImplementedError(f"Quant method is not yet supported: {quant_method!r}")/pass/g' llama.cpp/convert_hf_to_gguf.py

!python llama.cpp/convert_hf_to_gguf.py qwen2_unsloth_finetune \
    --outfile qwen2_model_receipt_f16.gguf --outtype f16


/kaggle/working
INFO:hf-to-gguf:Loading model: qwen2_unsloth_finetune
INFO:hf-to-gguf:Model architecture: Qwen2VLForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> F16, shape = {1536, 256}
INFO:hf-to

## Convert hf to gguf in f16 type

In [ ]:
%cd /kaggle/working/
!sed -i 's/raise NotImplementedError(f"Quant method is not yet supported: {quant_method!r}")/pass/g' llama.cpp/convert_hf_to_gguf.py

!python llama.cpp/convert_hf_to_gguf.py lora_merged_hf_gguf \
    --outfile qwen2_lora_receipt_f16.gguf --outtype f16 --mmproj


/kaggle/working
INFO:hf-to-gguf:Loading model: lora_merged_hf_gguf
INFO:hf-to-gguf:Model architecture: Qwen2VLForConditionalGeneration
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:v.blk.0.attn_out.bias,           torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:v.blk.0.attn_out.weight,         torch.bfloat16 --> F16, shape = {1280, 1280}
INFO:hf-to-gguf:v.blk.0.attn_q.bias,             torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:v.blk.0.attn_k.bias,             torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:v.blk.0.attn_v.bias,             torch.bfloat16 --> F32, shape = {1280}
INFO:hf-to-gguf:v.blk.0.attn_q.weight,           torch.bfloat16 --> F16, shape = {1280, 1280}
INFO:hf-to-gguf:v.blk.0.attn_k.weight,           torch.bfloat16 --> F16, shape = {1280, 1280}
INFO:hf-to-gguf:v.blk.0.attn_v.weight,           torch.bfloat16 -

## Convert f16 to Q4_K_M

In [ ]:
# Force le retour dans le dossier de travail
%cd /kaggle/working/

# 1. Prépare le dossier de build proprement
!cmake -S llama.cpp -B llama.cpp/build -DBUILD_SHARED_LIBS=OFF

# 2. Compile uniquement l'outil nécessaire (quantize)
!cmake --build llama.cpp/build --config Release --target llama-quantize -j 2


/kaggle/working
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- 

In [ ]:
!find llama.cpp -name "llama-quantize"


llama.cpp/build/bin/llama-quantize


In [ ]:
!./llama.cpp/build/bin/llama-quantize qwen2_receipt_f16.gguf qwen2_vision_Q4_K_M.gguf Q4_K_M


llama_print_build_info: build = 9010 (d05fe1d7d)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'qwen_model_receipt_f16.gguf' to 'qwen2_vision_Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 18 key-value pairs and 520 tensors from qwen_model_receipt_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = clip
llama_model_loader: - kv   1:                               general.type str              = mmproj
llama_model_loader: - kv   2:                               general.name str              = Qwen2_Merged_Hf_Gguf
llama_model_loader: - kv   3:                         general.size_label str              = 665M
llama_model_loader: - kv   4:                          general.file_type u32              = 1
llama_model_loader: - kv   5:                    clip.has_vision_e